In [1]:
import pandas as pd
import numpy as np
import pickle
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from src.neuralnet import NeuralNetwork
from src.layers import DenseLayer, DropoutLayer
from src.activation import ReLUActivation, SoftmaxActivation

In [2]:
import pandas as pd
import numpy as np
import pickle
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Célula 2: Preparação de Dados
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.strip()

# 1. Carregar o CSV
df = pd.read_csv("data/dataset-exemplos.csv", sep=";")

# 2. Garantir que só usamos as 5 classes oficiais do guião
CLASSES_OFICIAIS = ["Human", "Google", "Meta", "OpenAI", "Mistral"]

# Filtrar: manter apenas as linhas onde o 'Label' está dentro das oficiais
# (Isto apaga coisas como 'Anthropic' ou erros de digitação)
df = df[df["Label"].isin(CLASSES_OFICIAIS)].copy()

# 3. Criar a coluna 'clean_text' (AQUELA QUE FALTAVA!)
df["clean_text"] = df["Text"].apply(clean_text)

# 4. Forçar o LabelEncoder a conhecer APENAS as 5 classes oficiais
le = LabelEncoder()
le.fit(CLASSES_OFICIAIS) 

# Transformar as labels em números (0, 1, 2, 3, 4)
labels_idx = le.transform(df["Label"])

# Criar a matriz One-Hot Encoding (necessária para o framework NumPy)
y_onehot = np.zeros((len(labels_idx), len(le.classes_)))
y_onehot[np.arange(len(labels_idx)), labels_idx] = 1

# 5. Extração de Features (TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df["clean_text"]).toarray()

print(f"Número de textos válidos: {len(df)}")
print(f"Formato dos dados (X): {X.shape}")
print(f"Formato das labels (y): {y_onehot.shape}")
print(f"Classes reconhecidas pelo modelo: {le.classes_}")

Número de textos válidos: 102
Formato dos dados (X): (102, 2577)
Formato das labels (y): (102, 5)
Classes reconhecidas pelo modelo: ['Google' 'Human' 'Meta' 'Mistral' 'OpenAI']


In [3]:
net = NeuralNetwork(epochs=100, batch_size=32, learning_rate=0.005) # Menor LR, mais épocas

net.add(DenseLayer(256, input_shape=(X.shape[1],)))
net.add(ReLUActivation())
net.add(DropoutLayer(drop_rate=0.4)) # Aumentar o dropout para o modelo não decorar os textos
net.add(DenseLayer(128))
net.add(ReLUActivation())
net.add(DropoutLayer(drop_rate=0.2))
net.add(DenseLayer(len(le.classes_)))
net.add(SoftmaxActivation())

In [4]:
with open("modelo_numpy_artefactos.pkl", "wb") as f:
    pickle.dump({
        "model": net, 
        "vectorizer": vectorizer, 
        "label_encoder": le
    }, f)
print("Modelo Numpy guardado com sucesso!")

Modelo Numpy guardado com sucesso!
